# Coffee Leaf Disease Detection & Classification
**ETI 4210 Machine Learning Project — Dedan Kimathi University of Technology (DeKUT)**

### Project Team & Roles:
* **Beverlyne Tanui**: Data Collection
* **Blair Nduta**: Data Preparation
* **Bruce Boge**: Model Development
* **Hezron Nyakeruma**: Evaluation & Tuning
* **Kilavi Musyoki**: Deployment & Demo (Streamlit)

---
### Objective:
Empowering Kenyan smallholder coffee farmers to accurately detect and classify coffee leaf diseases from smartphone photographs.
* **Architecture**: **MobileNetV2** (Transfer Learning + Fine-Tuning)
* **Target Resolution**: 224 × 224 pixels (RGB)
* **Target Validation Accuracy**: 85%+
* **Evaluation**: Accuracy, Precision, Recall, F1-Score, and Confusion Matrix Heatmap
* **Target Deployment**: Lightweight Streamlit Web Application

## 1. Environment & Hardware Acceleration Setup
First, verify that Google Colab is using GPU hardware acceleration (e.g. **T4 GPU**). If not enabled, go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import os
import sys
import json
import time
import shutil
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Detected: {gpus[0].name} (Hardware Acceleration Active!)")
else:
    print("Notice: No GPU detected. In Colab, make sure to select Runtime > Change runtime type > T4 GPU.")

## 2. Dataset Setup & Google Drive Mounting
This cell automatically checks whether you are running in **Google Colab** or **locally**.
* In Colab, it mounts Google Drive and extracts your dataset zip file.
* Recommended setup: Upload `Coffee_leaf_Diseases.zip` to your Google Drive root. Unzipping it directly into Colab's local disk (`/content/dataset/`) provides 10x faster I/O than reading individual files from Drive.

In [ ]:
# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Path to zip file in Google Drive
    ZIP_PATH = '/content/drive/MyDrive/Coffee_leaf_Diseases.zip'
    DATA_DIR = '/content/dataset/Coffee leaf Diseases'
    
    if os.path.exists(ZIP_PATH) and not os.path.exists(DATA_DIR):
        print(f"Unzipping {ZIP_PATH} to /content/dataset ...")
        import zipfile
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall('/content/dataset')
        print("Dataset extracted successfully!")
    elif not os.path.exists(DATA_DIR):
        # Fallback to direct Google Drive directory if not zipped
        ALT_DIR = '/content/drive/MyDrive/Coffee leaf Diseases'
        if os.path.exists(ALT_DIR):
            DATA_DIR = ALT_DIR
else:
    # Local path when testing in VS Code
    DATA_DIR = 'Coffee leaf Diseases'

print(f"Using Dataset Directory: {DATA_DIR}")
assert os.path.exists(DATA_DIR), f"Dataset path '{DATA_DIR}' not found. Please verify folder location!"


## 3. Dataset Discovery & Class Distribution Analysis
We inspect the dataset folder, verify image counts per disease class, and visualize class imbalance.

In [ ]:
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
image_records = []

for class_name in sorted(os.listdir(DATA_DIR)):
    class_folder = os.path.join(DATA_DIR, class_name)
    if os.path.isdir(class_folder):
        files = [f for f in os.listdir(class_folder) if f.lower().endswith(valid_extensions)]
        for fname in files:
            image_records.append({
                'filepath': os.path.join(class_folder, fname),
                'label': class_name
            })

df = pd.DataFrame(image_records)
print(f"Total valid images found: {len(df):,}")
print("\nImages per class:")
class_counts = df['label'].value_counts()
print(class_counts)

# Visualizing class distribution
plt.figure(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
plt.title('Coffee Leaf Disease Dataset - Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=15)
for i, count in enumerate(class_counts.values):
    plt.text(i, count + 20, f"{count:,}", ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Stratified Dataset Splitting (80% Train, 10% Val, 10% Test)
To guarantee robust evaluation without data leakage, we partition the dataset with stratified sampling so all classes are proportionally represented in train, validation, and test sets.

In [ ]:
# 80% Train, 20% Temp (Val + Test)
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df['label']
)

# 50/50 split of Temp into Validation (10%) and Test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['label']
)

print(f"Training set:   {len(train_df):,} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation set: {len(val_df):,} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test set:       {len(test_df):,} images ({len(test_df)/len(df)*100:.1f}%)")

class_names = sorted(df['label'].unique().tolist())
num_classes = len(class_names)
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
idx_to_class = {idx: cls for idx, cls in enumerate(class_names)}

print(f"\nClasses ({num_classes}): {class_names}")


## 5. Image Preprocessing & Data Augmentation Pipeline
* **Input dimensions**: Standard `224 × 224 × 3` required by MobileNetV2.
* **Data Augmentation**: Rotation, horizontal flips, zooming, contrast adjustments, and translation to simulate diverse field angles and lighting conditions.
* **Preprocessing**: `tf.keras.applications.mobilenet_v2.preprocess_input` normalizes pixel values to `[-1, 1]`.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomContrast(0.1),
], name='data_augmentation')

def load_and_preprocess_image(path, label, is_training=False):
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)
    return img, label

def create_dataset(dataframe, is_training=False):
    paths = dataframe['filepath'].values
    labels = [class_to_idx[lbl] for lbl in dataframe['label'].values]
    labels = tf.keras.utils.to_categorical(labels, num_classes=num_classes)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    
    ds = ds.map(lambda p, l: load_and_preprocess_image(p, l, is_training), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    
    if is_training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
        
    ds = ds.prefetch(buffer_size=AUTOTUNE)
    return ds

train_ds = create_dataset(train_df, is_training=True)
val_ds = create_dataset(val_df, is_training=False)
test_ds = create_dataset(test_df, is_training=False)

print("Data pipelines built successfully with prefetching & GPU memory optimization!")


## 6. Sample Batch Verification
Verify that images and labels are loading properly.

In [ ]:
sample_images, sample_labels = next(iter(train_ds))

plt.figure(figsize=(14, 8))
for i in range(min(8, len(sample_images))):
    ax = plt.subplot(2, 4, i + 1)
    # De-normalize [-1, 1] back to [0, 1] for visual display
    disp_img = (sample_images[i].numpy() + 1.0) / 2.0
    disp_img = np.clip(disp_img, 0.0, 1.0)
    plt.imshow(disp_img)
    label_idx = np.argmax(sample_labels[i])
    plt.title(idx_to_class[label_idx], fontsize=11, fontweight='bold')
    plt.axis('off')

plt.suptitle('Augmented Training Batch (MobileNetV2 224x224)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Addressing Class Imbalance
As identified in Slide 10, classes with fewer images (e.g., Red Spider Mite) risk lower recall. We compute class weights using Scikit-Learn to penalize mistakes on minority classes proportionally.

In [ ]:
train_labels_idx = [class_to_idx[l] for l in train_df['label'].values]
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_idx),
    y=train_labels_idx
)
class_weights = dict(enumerate(weights))

print("Calculated Class Weights:")
for idx, weight in class_weights.items():
    print(f"  {idx_to_class[idx]}: {weight:.3f}")


## 8. MobileNetV2 Transfer Learning Architecture
* **Backbone**: MobileNetV2 pretrained on ImageNet (frozen initially).
* **Classification Head**:
  * GlobalAveragePooling2D: Flattens spatial features without parameter explosion.
  * BatchNormalization: Stabilizes internal activations.
  * Dense(128, ReLU): Learn task-specific coffee disease combinations.
  * Dropout(0.35): Mitigates overfitting.
  * Dense(num_classes, Softmax): Multi-class probability output.

In [ ]:
def build_mobilenet_model(num_classes, img_size=(224, 224, 3)):
    base_model = MobileNetV2(
        input_shape=img_size,
        include_top=False,
        weights='imagenet'
    )
    # Freeze backbone weights for initial feature extraction
    base_model.trainable = False
    
    inputs = layers.Input(shape=img_size, name='leaf_input')
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='global_pool')(x)
    x = layers.BatchNormalization(name='batch_norm')(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4), name='dense_features')(x)
    x = layers.Dropout(0.35, name='dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='prediction_head')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name='CoffeeLeaf_MobileNetV2')
    return model, base_model

model, base_model = build_mobilenet_model(num_classes=num_classes)
model.summary()


## 9. Phase 1 Training: Feature Extraction (Frozen Base)
We train only the classification head with learning rate `1e-3`.

In [ ]:
EPOCHS_PHASE1 = 12

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_coffee_model_phase1.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("Starting Phase 1 Training...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weights,
    callbacks=callbacks_phase1,
    verbose=1
)


## 10. Phase 2 Training: Fine-Tuning Pretrained Layers
We unfreeze top convolutional layers of MobileNetV2 (from layer 100 onwards) and continue training with a lower learning rate (`1e-5`) to specialize low-level feature extraction for coffee rust and spot patterns.

In [ ]:
# Unfreeze top layers of base model
base_model.trainable = True
fine_tune_at = 100

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Total layers in MobileNetV2: {len(base_model.layers)}")
print(f"Trainable layers for fine-tuning: {len(base_model.layers) - fine_tune_at}")

# Re-compile with small learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_PHASE2 = 12
callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    ModelCheckpoint('coffee_leaf_disease_mobilenetv2_final.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("\nStarting Phase 2 Training (Fine-Tuning)...")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weights,
    callbacks=callbacks_phase2,
    verbose=1
)


## 11. Performance Learning Curves
Plotting combined accuracy and loss curves for Phase 1 (Feature Extraction) and Phase 2 (Fine-Tuning).

In [ ]:
def plot_combined_history(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']
    
    epochs = range(1, len(acc) + 1)
    split_epoch = len(h1.history['accuracy'])
    
    plt.figure(figsize=(15, 6))
    
    # Accuracy Plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, label='Training Accuracy', color='#1f77b4', lw=2)
    plt.plot(epochs, val_acc, label='Validation Accuracy', color='#ff7f0e', lw=2)
    plt.axvline(x=split_epoch, color='gray', linestyle='--', label='Fine-Tuning Start')
    plt.title('Training and Validation Accuracy', fontsize=13, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    
    # Loss Plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, label='Training Loss', color='#1f77b4', lw=2)
    plt.plot(epochs, val_loss, label='Validation Loss', color='#ff7f0e', lw=2)
    plt.axvline(x=split_epoch, color='gray', linestyle='--', label='Fine-Tuning Start')
    plt.title('Training and Validation Loss', fontsize=13, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_validation_curves.png', dpi=300)
    plt.show()

plot_combined_history(history_phase1, history_phase2)


## 12. Model Evaluation on Held-Out Test Set
*(Deliverables for **Hezron Nyakeruma**)*:
* Accuracy & Loss on unseen test set (Target: **85%+**)
* Classification Report: Precision, Recall, F1-Score per disease class
* Seaborn Confusion Matrix Heatmap (Raw counts and normalized percentages)

In [ ]:
# Evaluate model performance on unseen test data
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"\n==========================================")
print(f"FINAL TEST ACCURACY: {test_acc * 100:.2f}%  (Target: 85%+)")
print(f"FINAL TEST LOSS:     {test_loss:.4f}")
print(f"==========================================\n")

# Compute predictions
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = []
for _, labels in test_ds:
    y_true.extend(np.argmax(labels.numpy(), axis=1))
y_true = np.array(y_true)

# Classification Report
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax[0])
ax[0].set_title('Confusion Matrix (Raw Counts)', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=ax[1])
ax[1].set_title('Normalized Confusion Matrix (%)', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Predicted Label')
ax[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('confusion_matrix_results.png', dpi=300)
plt.show()


## 13. Model Export for Deployment
*(Deliverables for **Kilavi Musyoki**)*:
* Model saved in Keras `.keras` format.
* Export `class_labels.json` for mapping predictions in the Streamlit application.
* Optional: Export lightweight TensorFlow Lite (`.tflite`) model for edge devices.

In [ ]:
# Save final Keras model
final_model_path = 'coffee_leaf_disease_mobilenetv2_final.keras'
model.save(final_model_path)
print(f"Saved trained model to: {final_model_path}")

# Save class mapping
labels_meta = {
    'classes': class_names,
    'class_to_idx': class_to_idx,
    'idx_to_class': idx_to_class,
    'img_size': list(IMG_SIZE)
}
with open('class_labels.json', 'w') as f:
    json.dump(labels_meta, f, indent=4)
print("Saved class labels mapping to: class_labels.json")

# Export TFLite model
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open('coffee_leaf_disease_model.tflite', 'wb') as f:
        f.write(tflite_model)
    print("Saved optimized edge model to: coffee_leaf_disease_model.tflite")
except Exception as e:
    print(f"TFLite notice: {e}")


## 14. Real-Time Inference Demo
Test the trained model on single leaf photographs.

In [ ]:
def predict_coffee_leaf(image_path_or_array, model, class_names):
    if isinstance(image_path_or_array, str):
        img_raw = Image.open(image_path_or_array).convert('RGB')
    else:
        img_raw = image_path_or_array
        
    img_resized = img_raw.resize(IMG_SIZE)
    img_array = np.array(img_resized)
    img_preprocessed = preprocess_input(img_array.astype(np.float32))
    img_batch = np.expand_dims(img_preprocessed, axis=0)
    
    preds = model.predict(img_batch, verbose=0)[0]
    top_idx = np.argmax(preds)
    top_class = class_names[top_idx]
    top_confidence = preds[top_idx] * 100
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].imshow(img_raw)
    status_color = 'green' if top_class == 'Healthy' else 'red'
    axes[0].set_title(f"Prediction: {top_class}\nConfidence: {top_confidence:.1f}%", fontsize=12, fontweight='bold', color=status_color)
    axes[0].axis('off')
    
    bar_colors = ['#2ca02c' if c == 'Healthy' else '#d62728' for c in class_names]
    axes[1].barh(class_names, preds * 100, color=bar_colors)
    axes[1].set_xlim(0, 100)
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title('Class Probabilities', fontsize=12, fontweight='bold')
    for i, v in enumerate(preds * 100):
        axes[1].text(v + 1, i, f"{v:.1f}%", va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    return top_class, top_confidence

# Demonstrate inference on 3 random test samples
sample_test = test_df.sample(min(3, len(test_df)), random_state=SEED)
for _, row in sample_test.iterrows():
    print(f"True Label: {row['label']}")
    predict_coffee_leaf(row['filepath'], model, class_names)
